# Ejercicio 1 — Fashion MNIST: ACP, t-SNE y UMAP
**Lead University · Minería de Datos · Tarea 5**

Comparación de ACP, t-SNE y UMAP sobre el dataset Fashion MNIST (`ropa.csv`):  
784 variables de píxeles, 1 000 imágenes, 10 tipos de prendas.

**Objetivo:** visualizar si los métodos de reducción dimensional separan los 10 tipos de prenda en 2D y 3D.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from scripts import DimReducer

print('Librerías cargadas.')

Librerías cargadas.


---
## a) Carga de datos

In [2]:
df = pd.read_csv('datos/ropa.csv')

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Mapear etiquetas numéricas a nombres de prenda
label_map = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}
df['label'] = df['label'].map(label_map)

print(f'Dimensiones: {df.shape}')
print(df['label'].value_counts().sort_index())

Dimensiones: (1000, 785)
label
Ankle boot     109
Bag            109
Coat            94
Dress           89
Pullover       111
Sandal          96
Shirt           91
Sneaker        102
T-shirt/top     89
Trouser        110
Name: count, dtype: int64


**Contexto del dataset:**  
Fashion MNIST contiene imágenes de 28×28 píxeles (784 variables) de 10 tipos de prendas.
Con 784 dimensiones, el ACP lineal captura poca estructura visual; t-SNE y UMAP, al ser no lineales,
deben separar mejor las clases.

---
## b) ACP, t-SNE y UMAP en 2 componentes

### Exploración de n_neighbors para UMAP

In [3]:
dr = DimReducer(df, color_col='label', seed=42)

dr.explore_umap_neighbors(neighbors_list=[5, 15, 30, 50])

**Selección de n_neighbors:**  
Se probaron los valores 5, 15, 30 y 50. Con n_neighbors=5 los clústeres se fragmentan en
micro-grupos pequeños sin cohesión global — Trouser y calzado quedan separados pero el resto
se dispersa en pedazos inconexos. Con n_neighbors=15 los grupos ganan cohesión y la separación
entre categorías es más nítida: Trouser, Bag y el calzado forman regiones diferenciadas con
bordes más claros. Con n_neighbors=30 la estructura se vuelve una cinta continua donde los
grupos quedan encadenados, reduciendo la separación discreta entre clases. Con n_neighbors=50
los grupos se difuminan y mezclan aún más. **Se selecciona n_neighbors = 15** como el valor
que produce el mejor balance entre cohesión interna de cada categoría y separación entre clases.

In [ ]:
dr.fit(n_components=2, tsne_perplexity=30, umap_n_neighbors=15)

In [5]:
dr.plot_mapa_interactivo(dr.coords_pca, title='Fashion MNIST — ACP (2D)')
dr.plot_mapa_interactivo(dr.coords_tsne, title='Fashion MNIST — t-SNE (2D)')
dr.plot_mapa_interactivo(dr.coords_umap, title='Fashion MNIST — UMAP (2D)')

In [6]:
dr.plot_comparacion()

---
## c) ¿Cuál método separa mejor en 2D?

**ACP:** Aunque es un método lineal, las dos primeras componentes capturan suficiente varianza
para mostrar agrupaciones parciales: Trouser queda claramente separado hacia abajo-izquierda,
y el calzado (Ankle boot, Sneaker, Sandal) tiende a concentrarse en la zona superior-izquierda.
Sin embargo, las prendas de torso (T-shirt/top, Shirt, Pullover, Coat, Dress) se solapan
considerablemente en el centro del plano, limitando la utilidad del ACP para distinguir esas categorías.

**t-SNE:** Produce la separación más clara en esta ejecución. La mayoría de las 10 categorías
forman clústeres visiblemente delimitados y espacialmente separados entre sí. El solapamiento
existe pero es minoritario y se concentra, como se esperaba, entre las prendas de torso con
siluetas similares (Shirt, Pullover, Coat, T-shirt/top), cuyas texturas de píxeles son
estructuralmente parecidas.

**UMAP:** Con n_neighbors=15 produce una separación notablemente mejor que con valores más
altos. Trouser (morado) queda claramente separado en la zona inferior, Bag (azul) forma un
clúster aislado a la derecha, y el calzado se agrupa en la zona central-inferior. Sin embargo,
las prendas de torso (Shirt, Pullover, Coat, T-shirt/top) permanecen mezcladas en el centro,
por lo que la separación global es menos nítida que en t-SNE.

**Conclusión:** En esta ejecución, **t-SNE supera a UMAP en separación visual de clases**,
aunque la diferencia es menor con n_neighbors=15 que con valores más altos. Ambos superan al
ACP, que solo logra separar parcialmente las categorías con mayor diferencia estructural
(pantalones y calzado).

---
## d) ACP, t-SNE y UMAP en 3 componentes

In [ ]:
dr.fit(n_components=3, tsne_perplexity=30, umap_n_neighbors=15)

In [8]:
dr.plot_3d(dr.coords_pca, title='Fashion MNIST — ACP (3D)')
dr.plot_3d(dr.coords_tsne, title='Fashion MNIST — t-SNE (3D)')
dr.plot_3d(dr.coords_umap, title='Fashion MNIST — UMAP (3D)')

---
## e) ¿Cuál método da mejores resultados en 3D?

**ACP en 3D:** La nube de puntos queda prácticamente aplanada en el plano XY con muy poco
uso del eje Z. Los colores siguen completamente mezclados y no se observa ninguna agrupación
por categoría. La tercera componente no aporta separación útil.

**t-SNE en 3D:** Al igual que en la ejecución anterior, la tercera dimensión no se aprovecha
— los puntos quedan aplanados con escaso rango en Z. La separación no mejora respecto a 2D
y en algunos ángulos resulta más difícil de interpretar que el mapa plano.

**UMAP en 3D:** Es el método que mejor aprovecha la tercera dimensión con n_neighbors=15.
El calzado (Ankle boot, Sneaker) queda claramente separado en la zona superior, Trouser
(morado) se ubica en la zona inferior, y Bag (azul) forma un grupo diferenciado en el centro.
La estructura 3D es más informativa que la proyección 2D de UMAP y supera a t-SNE en
separación discreta de clústeres.

**Conclusión:** En 3D, **UMAP supera a t-SNE**, invirtiendo el resultado de 2D. t-SNE no
traslada su ventaja bidimensional a la tercera dimensión porque su optimización tiende a
concentrar la estructura en los dos primeros ejes. UMAP, en cambio, distribuye la información
en las tres dimensiones, logrando una separación más clara de las categorías con perfiles
de píxeles más distintos. ACP sigue siendo el peor de los tres al no capturar estructura
no lineal en ninguna dimensión.